# 🔬 Soil GSD: Physical Scale Invariant Ridge Baseline

This notebook implements the ultimate domain-specific hack for camera-based grain size estimation: **Physical Scale Normalization**.

### The Core Flaw of Standard CNNs:
Standard image pipelines use `transforms.Resize((224, 224))`. This stretches or squashes images, forcing a 10cm stone and a 1mm sand grain to look identical in pixel space.

### The Solution:
1. **PPM Dynamic Rescaling:** We resize every image based on its unique Pixels Per Millimeter (PPM) value so that 1 pixel equals the exact same physical distance across all devices.
2. **Physical CenterCrop:** Instead of squashing, we take a `CenterCrop(224)`. This guarantees the CNN inspects the exact same physical area of soil (e.g., a 15mm x 15mm patch) for every single sample.

## Setup

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import random

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_DIR = "/kaggle/input/competitions/soil-grain-size-from-photos"
TRAIN_LABELS_PATH = os.path.join(DATA_DIR, "Training_labels_without_H374.csv")
PPM_PATH = os.path.join(DATA_DIR, "ppm.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "Training-All_Photos_without_H374")
TEST_IMG_DIR = os.path.join(DATA_DIR, "Test_All_Photos")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

Using device: cpu


## Path Mapping

In [2]:
def build_image_map(directory):
    img_map = {}
    if not os.path.exists(directory):
        return img_map
    for root, _, files in os.walk(directory):
        for file in files:
            name_idx = file.rfind('.')
            if name_idx != -1:
                sample_id = file[:name_idx].lower()
                img_map[sample_id] = os.path.join(root, file)
    return img_map

train_img_map = build_image_map(TRAIN_IMG_DIR)
test_img_map = build_image_map(TEST_IMG_DIR)

## EfficientNet Feature Extractor

In [3]:
encoder = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
feature_extractor = nn.Sequential(
    encoder.features,
    encoder.avgpool,
    nn.Flatten()
)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# CRITICAL FIX: We use CenterCrop instead of Resize to preserve the true physical scale!
img_transform = transforms.Compose([
    transforms.CenterCrop(224), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 118MB/s] 


## Dynamic Rescale & Feature Union Pipeline

In [4]:
def normalize_string(s):
    s = str(s).lower().strip()
    s = s.replace('ü', 'ue').replace('ö', 'oe').replace('ä', 'ae').replace('ß', 'ss')
    s = s.replace(',', '_').replace('-', '_').replace(' ', '_')
    while '__' in s:
        s = s.replace('__', '_')
    return s

def extract_hybrid_features(df, img_map, ppm_lookup, global_mean_ppm, set_name="Dataset"):
    features_list = []
    print(f"\n--- Running Scale-Invariant Extraction for {set_name} ---")
    
    # Target scale: we want all images normalized to a standard 15.0 pixels per millimeter
    TARGET_PPM = 15.0 
    
    for idx, row in df.iterrows():
        sample_id = str(row["sample_id"])
        norm_sample_id = normalize_string(sample_id)
        
        matching_paths = []
        for key, path in img_map.items():
            norm_key = normalize_string(key)
            if (norm_sample_id in norm_key) or (norm_key in norm_sample_id) or (norm_sample_id.replace('hpc_', '') in norm_key):
                matching_paths.append(path)
                
        feat = {"sample_id": row["sample_id"]}
        
        if matching_paths:
            all_embs = []
            all_edges = []
            all_r_means = []
            all_g_means = []
            all_b_means = []
            
            for img_path in matching_paths:
                img = cv2.imread(img_path)
                if img is not None:
                    h, w = img.shape[0], img.shape[1]
                    current_ppm = ppm_lookup.get((w, h), global_mean_ppm)
                    
                    # 1. DYNAMIC PHYSICAL RESCALE
                    scale_factor = TARGET_PPM / current_ppm
                    new_w = int(w * scale_factor)
                    new_h = int(h * scale_factor)
                    img_scaled = cv2.resize(img, (new_w, new_h))
                    
                    # Defensive fallback if the scaled image patch is smaller than crop window
                    if img_scaled.shape[0] < 224 or img_scaled.shape[1] < 224:
                        img_scaled = cv2.resize(img_scaled, (224, 224))
                    
                    # 2. Extract Classical CV Features from scaled frame
                    gray = cv2.cvtColor(img_scaled, cv2.COLOR_BGR2GRAY)
                    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
                    edges = cv2.Canny(blurred, 30, 100)
                    all_edges.append(np.mean(edges > 0))
                    
                    img_rgb = cv2.cvtColor(img_scaled, cv2.COLOR_BGR2RGB)
                    all_r_means.append(np.mean(img_rgb[:, :, 0]))
                    all_g_means.append(np.mean(img_rgb[:, :, 1]))
                    all_b_means.append(np.mean(img_rgb[:, :, 2]))
                    
                    # 3. Extract Deep Textures via CenterCrop
                    pil_img = Image.fromarray(img_rgb)
                    tensor_img = img_transform(pil_img).unsqueeze(0).to(device)
                    
                    with torch.no_grad():
                        cnn_emb = feature_extractor(tensor_img).cpu().numpy().flatten()
                    all_embs.append(cnn_emb)
            
            if all_embs:
                feat["edge_density"] = np.mean(all_edges)
                feat["r_mean"] = np.mean(all_r_means)
                feat["g_mean"] = np.mean(all_g_means)
                feat["b_mean"] = np.mean(all_b_means)
                feat["photo_count"] = len(all_embs)
                
                avg_emb = np.mean(all_embs, axis=0)
                for i, val in enumerate(avg_emb):
                    feat[f"cnn_{i}"] = val
            else:
                feat["edge_density"] = 0.0
                feat["r_mean"] = 127.0; feat["g_mean"] = 127.0; feat["b_mean"] = 127.0
                feat["photo_count"] = 0
        else:
            feat["edge_density"] = 0.0
            feat["r_mean"] = 127.0; feat["g_mean"] = 127.0; feat["b_mean"] = 127.0
            feat["photo_count"] = 0
            
        features_list.append(feat)
        
    return pd.DataFrame(features_list)

df_train_labels = pd.read_csv(TRAIN_LABELS_PATH)
df_sub = pd.read_csv(SAMPLE_SUB_PATH)
df_ppm = pd.read_csv(PPM_PATH)

ppm_lookup = df_ppm.set_index(["width", "height"])["ppm"].to_dict()
global_mean_ppm = df_ppm["ppm"].mean()

X_train_df = extract_hybrid_features(df_train_labels, train_img_map, ppm_lookup, global_mean_ppm, "Train Set")
X_test_df = extract_hybrid_features(df_sub, test_img_map, ppm_lookup, global_mean_ppm, "Test Set")


--- Running Scale-Invariant Extraction for Train Set ---

--- Running Scale-Invariant Extraction for Test Set ---


## Stable Direct Cumulative RidgeCV Training

In [5]:
target_cols = [col for col in df_train_labels.columns if col != "sample_id"]

common_cols = list(set(X_train_df.columns) & set(X_test_df.columns))
feature_cols = [col for col in common_cols if col != "sample_id"]

X = X_train_df[feature_cols].fillna(0)
Y = df_train_labels[target_cols].values # Back to super stable direct cumulative target
X_test = X_test_df[feature_cols].fillna(0)

print(f"➔ Scale-normalized features dimension: {X.shape[1]} columns aligned.")

# Scaled Pipeline with RidgeCV tracking direct cumulative curves
base_ridge_pipeline = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=np.logspace(-1, 5, 35))
)
model = MultiOutputRegressor(base_ridge_pipeline)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(Y.shape)
test_preds = np.zeros((len(X_test), Y.shape[1]))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, Y)):
    X_tr, Y_tr = X.iloc[train_idx], Y[train_idx]
    X_va, Y_va = X.iloc[val_idx], Y[val_idx]
    
    model.fit(X_tr, Y_tr)
    
    val_pred = model.predict(X_va)
    oof_preds[val_idx] = val_pred
    test_preds += model.predict(X_test) / kf.n_splits
    
    print(f"Fold {fold+1} MAE: {mean_absolute_error(Y_va, val_pred):.4f}")

print(f"\n➔ Overall Out-of-Fold Grand MAE: {mean_absolute_error(Y, oof_preds):.4f}")

➔ Scale-normalized features dimension: 1285 columns aligned.
Fold 1 MAE: 9.3095
Fold 2 MAE: 9.5904
Fold 3 MAE: 10.8117
Fold 4 MAE: 12.3699
Fold 5 MAE: 6.4746

➔ Overall Out-of-Fold Grand MAE: 9.7112


## Post-Processing

In [6]:
# Apply physical boundary limits and guarantee non-decreasing monotony
final_test_preds = np.clip(test_preds, 0.0, 100.0)
final_test_preds = np.sort(final_test_preds, axis=1)

submission = pd.DataFrame(final_test_preds, columns=target_cols)
submission.insert(0, "sample_id", df_sub["sample_id"])

submission.to_csv("submission.csv", index=False)
print("\n➔ Scale-Invariant submission.csv generated!")
display(submission.head())


➔ Scale-Invariant submission.csv generated!


,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,0.000000,0.000000,0.732306,22.110552,44.214969,52.489180,61.937474,76.937564,92.214586,99.300502,100.0
1,HPC_Audorfring,0.000000,0.388402,5.620659,27.290605,42.554718,50.845322,57.505792,71.610948,91.576909,99.316067,100.0
2,HPC_Muenster_BS6_9_0-10m,0.000000,0.000000,0.000000,0.000000,16.178299,23.818596,33.437842,56.135370,88.347888,98.884707,100.0
3,HPC_Airbus BS10-4bis7,2.010805,2.798181,7.966850,32.059987,60.739074,71.240945,78.965966,88.692420,94.728863,99.685226,100.0
4,HPC_Airbus BS12-5bis8,2.511712,2.993872,5.762687,31.159914,58.659356,65.542482,72.614614,83.193926,93.573356,99.507494,100.0
